### setup

In [1]:
import os
import torch
import json 
from unsloth import FastLanguageModel
from trl import SFTTrainer , SFTConfig 
from datasets import Dataset

os.environ["HF_HUB_DISABLE_XET"] = "1"

os.environ["UNSLOTH_DISABLE_FUSED_XET"] = "1"

print("library loaded")
print(f"torch version:{torch.__version__}")
print(f"cuda version:{torch.cuda.is_available()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\HP\anaconda3\envs\unsloth-ft\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0624 14:57:39.458000 3336 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0624 14:57:39.668000 3336 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


library loaded
torch version:2.12.0+cu130
cuda version:True


### LOAD MODLE 

In [2]:
model_path = r"c:\models\gemma2"

print("modle loaded... ", model_path) 

model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path ,
    device_map="auto",
    dtype = None,
    max_seq_length = 2048 , 
    load_in_4bit= True , 
)

print("modle loaded... ")
print(f"device:{model.device}")
print(f"dtype:{model.dtype}")

modle loaded...  c:\models\gemma2
==((====))==  Unsloth 2026.6.1: Fast Gemma patching. Transformers: 5.11.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 164/164 [00:07<00:00, 21.96it/s]
Unsloth: Will load c:\models\gemma2 as a legacy tokenizer.


modle loaded... 
device:cuda:0
dtype:torch.bfloat16


### LOAD MODEL FOR (LORA) PART 

In [3]:
print(":arrows_counterclockwise:get ready for LORA...")

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print(":white_check_mark:LORA is ready")
print(f":white_check_mark:ready parameters for training...: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}")



:arrows_counterclockwise:get ready for LORA...


Unsloth 2026.6.1 patched 18 layers with 18 QKV layers, 18 O layers and 18 MLP layers.


:white_check_mark:LORA is ready
:white_check_mark:ready parameters for training...: 19.61
